<a href="https://colab.research.google.com/github/JoudAlrubaish/technical-support-agent/blob/main/notebooks/02_model_b_extractive_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Model B — Technical Extractive QA**

##1. Environment & Dataset Loading

In [1]:
#imports and Hugging Face login
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [22]:
# load dataset
from datasets import load_dataset
import pandas as pd
import re

raw_dataset = load_dataset("UmerSajid/IT-Troubleshooting-Dataset", split="train")

raw_dataset

README.md:   0%|          | 0.00/3.56k [00:00<?, ?B/s]

(…)roubleshooting_Dataset_with_Links%20.csv:   0%|          | 0.00/4.46M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10500 [00:00<?, ? examples/s]

Dataset({
    features: ['ID', 'Category', 'Issue', 'Symptoms', 'Solution Steps', 'Severity', 'Estimated Resolution Time', 'Common Causes', 'Keywords', 'Urdu Solution', 'Documentation Link'],
    num_rows: 10500
})

In [23]:
#convert to pandas DataFrame
df = raw_dataset.to_pandas()

print("Total rows:", len(df))
print(df.columns.tolist())


Total rows: 10500
['ID', 'Category', 'Issue', 'Symptoms', 'Solution Steps', 'Severity', 'Estimated Resolution Time', 'Common Causes', 'Keywords', 'Urdu Solution', 'Documentation Link']


In [24]:
# remove artificial variant numbers from issue names to make the dataset clean

df["issue_base"] = (df["Issue"].str.replace(
        r"\s*-\s*Variant\s*\d+\s*$",
        "",
        regex=True
    )
    .str.strip()
)

unique_cases = (df.drop_duplicates(subset=["issue_base"]).reset_index(drop=True))

print("Original rows:", len(df))
print("Unique IT issues:", len(unique_cases))

Original rows: 10500
Unique IT issues: 33


In [25]:
#inspect the real cases
pd.set_option("display.max_colwidth", 300)

unique_cases[
    [
        "Category",
        "issue_base",
        "Symptoms",
        "Solution Steps"
    ]
].sample(n=min(12, len(unique_cases)),random_state=42)

,Category,issue_base,Symptoms,Solution Steps
31,Microsoft Office,Outlook not sending emails,Emails stuck in Outbox,"Check internet connection, clear Outbox, repair email account settings"
15,Hardware,Laptop overheating,"Fan running loud, sudden shutdowns","Clean vents, apply thermal paste, use a cooling pad"
26,MAC,AirDrop not working,Devices not visible or unable to send files,"Check Bluetooth and Wi-Fi, ensure devices are discoverable, restart devices"
17,Software,Browser not loading pages,"Blank pages, 'site can't be reached' errors","Clear cache, disable extensions, reset browser settings"
8,Security,Firewall blocking app,App cannot access the internet,"Check firewall rules, create exception for the app, restart firewall service"
9,Network,Wi-Fi not connecting,"No internet, error messages","Restart router, reset adapter, check driver settings"
19,Software,Application not updating,Update fails or stalls,"Check internet connection, restart application, reinstall application"
21,System,System running slow,"Lagging, high CPU usage","Close unnecessary programs, upgrade RAM, run disk cleanup"
12,Network,VPN not connecting,Cannot establish a secure connection,"Check VPN credentials, restart VPN service, check firewall settings"
0,Cloud Computing,AWS instance not starting,Instance stuck in 'pending' state,"Check instance status in AWS console, review security group settings, restart instance"


## 2. Extractive QA Dataset Preparation

The original dataset contains many artificial variants of the same IT
problems, to avoid duplicated and highly similar examples, only the 33
unique troubleshooting issues are retained.

Each unique issue is converted into three extractive QA examples:

1. Symptoms of the issue.
2. Common causes of the issue.
3. Resolution steps.

This produces a compact and diverse QA dataset while keeping every
answer directly grounded in the support context.

In [28]:
#create the context and QA pairs
qa_examples = []

for _, row in unique_cases.iterrows():

    issue = str(row["issue_base"]).strip()
    symptoms = str(row["Symptoms"]).strip()
    solution = str(row["Solution Steps"]).strip()

    # Trusted context for the issue.
    context = (
        f"Issue: {issue}\n"
        f"Symptoms: {symptoms}\n"
        f"Solution Steps: {solution}"
    )

    # Q1 — Symptoms
    qa_examples.append({
        "issue": issue,
        "question": f"What are the symptoms of {issue}?",
        "context": context,
        "answer_text": symptoms,
        "answer_start": context.find(symptoms)
    })

    # Q2 — Resolution
    qa_examples.append({
        "issue": issue,
        "question": f"How can {issue} be resolved?",
        "context": context,
        "answer_text": solution,
        "answer_start": context.find(solution)
    })

In [30]:
#convert it to pandas DataFrame
#We have 66 QA pairs from 33 distinct issues, which is above the project minimum of 30 QA pairs
qa_df = pd.DataFrame(qa_examples)

print("Unique IT issues:", qa_df["issue"].nunique())
print("Total QA pairs:", len(qa_df))

qa_df[["issue", "question", "answer_text"]].sample(12,random_state=42)

Unique IT issues: 33
Total QA pairs: 66


,issue,question,answer_text
54,Spinning beach ball,What are the symptoms of Spinning beach ball?,System unresponsive with spinning cursor
62,Outlook not sending emails,What are the symptoms of Outlook not sending emails?,Emails stuck in Outbox
0,AWS instance not starting,What are the symptoms of AWS instance not starting?,Instance stuck in 'pending' state
45,Disk space running out,How can Disk space running out be resolved?,"Delete temporary files, uninstall unused programs, move files to external storage"
5,Google Cloud storage access error,How can Google Cloud storage access error be resolved?,"Check IAM permissions, verify bucket policies, authenticate properly"
63,Outlook not sending emails,How can Outlook not sending emails be resolved?,"Check internet connection, clear Outbox, repair email account settings"
16,Firewall blocking app,What are the symptoms of Firewall blocking app?,App cannot access the internet
12,Antivirus not updating,What are the symptoms of Antivirus not updating?,Definitions out of date
65,PowerPoint presentation not saving,How can PowerPoint presentation not saving be resolved?,"Check file permissions, save to a different location, repair Office installation"
30,Laptop overheating,What are the symptoms of Laptop overheating?,"Fan running loud, sudden shutdowns"


## 3. Train / Validation / Test Split

The dataset is split by unique IT issue rather than by individual QA pair, which prevents different questions about the same troubleshooting issue
from appearing in both training and evaluation data, reducing data leakage.

In [31]:
from sklearn.model_selection import train_test_split

# get the 33 unique IT issues
issues = qa_df["issue"].unique()

# 80% train issues, 20% temporary issues
train_issues, temp_issues = train_test_split(
    issues,
    test_size=0.20,
    random_state=42
)

# split the temporary issues equally into validation and testing
val_issues, test_issues = train_test_split(
    temp_issues,
    test_size=0.50,
    random_state=42
)

In [32]:
#create the three QA datasets
qa_train_df = qa_df[qa_df["issue"].isin(train_issues)].reset_index(drop=True)
qa_val_df = qa_df[qa_df["issue"].isin(val_issues)].reset_index(drop=True)
qa_test_df = qa_df[qa_df["issue"].isin(test_issues)].reset_index(drop=True)

print("Train issues:", qa_train_df["issue"].nunique())
print("Train QA pairs:", len(qa_train_df))

print("\nValidation issues:", qa_val_df["issue"].nunique())
print("Validation QA pairs:", len(qa_val_df))

print("\nTest issues:", qa_test_df["issue"].nunique())
print("Test QA pairs:", len(qa_test_df))

Train issues: 26
Train QA pairs: 52

Validation issues: 3
Validation QA pairs: 6

Test issues: 4
Test QA pairs: 8


In [33]:
#data leakage check
print("Train - Validation overlap:", len(set(train_issues) & set(val_issues)))
print("Train - Test overlap:",len(set(train_issues) & set(test_issues)))
print("Validation - Test overlap:",len(set(val_issues) & set(test_issues)))

Train - Validation overlap: 0
Train - Test overlap: 0
Validation - Test overlap: 0


## 4. Tokenization & Long-Context Preprocessing

The QA examples are tokenized using DistilBERT.

Long contexts are handled using a sliding-window strategy with:

- Maximum sequence length: 384 tokens
- Document stride: 96 tokens

Character-level answer positions are converted into token-level
start and end positions required for extractive QA training.

In [57]:
#load tokenizer
from transformers import AutoTokenizer

MODEL_B = "distilbert-base-uncased"

tokenizer_b = AutoTokenizer.from_pretrained(MODEL_B)
MAX_LENGTH = 384
DOC_STRIDE = 96

In [58]:
#convert DataFrames to Hugging Face Dataset
from datasets import Dataset

qa_train = Dataset.from_pandas(
    qa_train_df,
    preserve_index=False
)

qa_val = Dataset.from_pandas(
    qa_val_df,
    preserve_index=False
)

qa_test = Dataset.from_pandas(
    qa_test_df,
    preserve_index=False
)

print(qa_train)
print(qa_val)
print(qa_test)

Dataset({
    features: ['issue', 'question', 'context', 'answer_text', 'answer_start'],
    num_rows: 52
})
Dataset({
    features: ['issue', 'question', 'context', 'answer_text', 'answer_start'],
    num_rows: 6
})
Dataset({
    features: ['issue', 'question', 'context', 'answer_text', 'answer_start'],
    num_rows: 8
})


In [66]:
#QA preprocessing function
def prepare_qa_features(examples):
    """
    Tokenize question-context pairs and convert character-level
    answer spans into token-level start and end positions.

    Long contexts are split into overlapping windows using a
    document stride.
    """

    tokenized = tokenizer_b(
        [q.strip() for q in examples["question"]],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop(
        "overflow_to_sample_mapping"
    )

    offsets = tokenized.pop(
        "offset_mapping"
    )

    start_positions = []
    end_positions = []

    for feature_index, feature_offsets in enumerate(offsets):

        input_ids = tokenized["input_ids"][feature_index]

        cls_index = input_ids.index(
            tokenizer_b.cls_token_id
        )

        sequence_ids = tokenized.sequence_ids(
            feature_index
        )

        sample_index = sample_mapping[
            feature_index
        ]

        answer_start = examples["answer_start"][
            sample_index
        ]

        answer_text = examples["answer_text"][
            sample_index
        ]

        answer_end = (
            answer_start
            + len(answer_text)
        )

        # find the beginning of the context tokens
        context_start = 0

        while sequence_ids[context_start] != 1:
            context_start += 1

        # find the end of the context tokens
        context_end = len(sequence_ids) - 1

        while sequence_ids[context_end] != 1:
            context_end -= 1

        # if the answer is outside this window point to the CLS token
        if (
            feature_offsets[context_start][0] > answer_start
            or
            feature_offsets[context_end][1] < answer_end
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # find answer start token
        token_start = context_start

        while (
            token_start <= context_end
            and feature_offsets[token_start][1] <= answer_start
        ):
            token_start += 1

        # find answer end token
        token_end = context_end

        while (
            token_end >= context_start
            and feature_offsets[token_end][0] >= answer_end
        ):
            token_end -= 1

        start_positions.append(token_start)
        end_positions.append(token_end)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized

In [67]:
#tokenize the three splits
qa_train_features = qa_train.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_train.column_names
)

qa_val_features = qa_val.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_val.column_names
)

qa_test_features = qa_test.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_test.column_names
)

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

In [61]:
#checking
print("Train features:", len(qa_train_features))
print("Validation features:", len(qa_val_features))
print("Test features:", len(qa_test_features))
print("\nColumns:" , qa_train_features.column_names)


Train features: 52
Validation features: 6
Test features: 8

Columns: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions']


## 5. Baseline Evaluation

Before fine-tuning, the pretrained DistilBERT model is evaluated on the
held-out test set.

In [41]:
#load the baseline QA model
import torch
from transformers import AutoModelForQuestionAnswering

baseline_model_b = AutoModelForQuestionAnswering.from_pretrained(MODEL_B)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

baseline_model_b.to(device)
baseline_model_b.eval()

print("Device:", device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda


In [42]:
#Exact Match and F1 functions
import re
import string
from collections import Counter


def normalize_answer(text):
    """
    Normalize text before computing Exact Match and token-level F1.
    """

    text = text.lower()

    text = "".join(
        char for char in text
        if char not in string.punctuation
    )

    text = re.sub(
        r"\b(a|an|the)\b",
        " ",
        text
    )

    return " ".join(text.split())


def exact_match_score(prediction, truth):
    """
    Return 1 when normalized prediction exactly matches the reference.
    """

    return int(
        normalize_answer(prediction)
        == normalize_answer(truth)
    )


def token_f1_score(prediction, truth):
    """
    Compute token-level F1 between prediction and reference answer.
    """

    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(truth).split()

    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return float(pred_tokens == truth_tokens)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)

    return (
        2 * precision * recall
        / (precision + recall)
    )

In [43]:
#prediction function

def predict_answer(model, question, context):
    """
    Extract the most likely answer span from the supplied context.
    """

    inputs = tokenizer_b(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=MAX_LENGTH,
        return_offsets_mapping=True
    )

    offsets = inputs.pop("offset_mapping")[0]
    sequence_ids = inputs.sequence_ids(0)

    # DistilBERT does not need token_type_ids.
    inputs.pop("token_type_ids", None)

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits[0].cpu()
    end_logits = outputs.end_logits[0].cpu()

    # Only allow answer spans from the context.
    context_indices = [
        i
        for i, sequence_id in enumerate(sequence_ids)
        if sequence_id == 1
    ]

    best_score = float("-inf")
    best_start = None
    best_end = None

    MAX_ANSWER_TOKENS = 50

    for start_idx in context_indices:

        max_end = min(
            start_idx + MAX_ANSWER_TOKENS,
            context_indices[-1] + 1
        )

        for end_idx in range(start_idx, max_end):

            if sequence_ids[end_idx] != 1:
                continue

            score = (
                start_logits[start_idx].item()
                + end_logits[end_idx].item()
            )

            if score > best_score:
                best_score = score
                best_start = start_idx
                best_end = end_idx

    if best_start is None:
        return ""

    char_start = offsets[best_start][0]
    char_end = offsets[best_end][1]

    return context[char_start:char_end]


In [44]:
#evaluate baseline on the 8 unseen test questions
baseline_predictions = []

for _, row in qa_test_df.iterrows():

    prediction = predict_answer(
        baseline_model_b,
        row["question"],
        row["context"]
    )

    baseline_predictions.append({
        "issue": row["issue"],
        "question": row["question"],
        "reference": row["answer_text"],
        "prediction": prediction,

        "exact_match": exact_match_score(
            prediction,
            row["answer_text"]
        ),

        "f1": token_f1_score(
            prediction,
            row["answer_text"]
        )
    })

In [45]:
#show the model results
baseline_results_df = pd.DataFrame(baseline_predictions)

baseline_em = baseline_results_df["exact_match"].mean()
baseline_f1 = baseline_results_df["f1"].mean()

print("BASELINE RESULTS")
print("-------------------------")
print(f"Exact Match : {baseline_em:.4f}")
print(f"Token F1    : {baseline_f1:.4f}")

BASELINE RESULTS
-------------------------
Exact Match : 0.0000
Token F1    : 0.4162


In [46]:
#show the predictions
baseline_results_df[
    [
        "question",
        "reference",
        "prediction",
        "exact_match",
        "f1"
    ]
]

,question,reference,prediction,exact_match,f1
0,What are the symptoms of Wi-Fi not connecting?,"No internet, error messages",Symptoms: No internet,0,0.571429
1,How can Wi-Fi not connecting be resolved?,"Restart router, reset adapter, check driver settings",Issue: Wi-Fi not connecting\nSymptoms: No internet,0,0.000000
2,What are the symptoms of Laptop overheating?,"Fan running loud, sudden shutdowns","Symptoms: Fan running loud, sudden shutdowns\nSolution Steps: Clean vents, apply thermal paste",0,0.555556
3,How can Laptop overheating be resolved?,"Clean vents, apply thermal paste, use a cooling pad","Symptoms: Fan running loud, sudden shutdowns\nSolution Steps: Clean vents, apply thermal paste",0,0.476190
4,What are the symptoms of AirDrop not working?,Devices not visible or unable to send files,"unable to send files\nSolution Steps: Check Bluetooth and Wi-Fi, ensure",0,0.421053
5,How can AirDrop not working be resolved?,"Check Bluetooth and Wi-Fi, ensure devices are discoverable, restart devices","files\nSolution Steps: Check Bluetooth and Wi-Fi, ensure",0,0.555556
6,What are the symptoms of Outlook not sending emails?,Emails stuck in Outbox,Emails stuck in Out,0,0.750000
7,How can Outlook not sending emails be resolved?,"Check internet connection, clear Outbox, repair email account settings",Symptoms: Emails stuck in Out,0,0.000000


**Findings:** before fine-tuning the model sometimes finds a partially overlapping span, but none of the 8 answers are exactly correct.

## 6. Model Fine-Tuning


In [69]:
#initialize a fresh QA model
from transformers import DefaultDataCollator
data_collator = DefaultDataCollator()

from transformers import AutoModelForQuestionAnswering, set_seed

set_seed(42)

model_b_exp2 = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_B
)

model_b_exp2.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

In [70]:
#training config
from transformers import TrainingArguments

training_args_b_exp2 = TrainingArguments(
    output_dir="models/qa_model_exp2",

    learning_rate=3e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_strategy="epoch",

    report_to="none",
    seed=42
)

In [71]:
#create trainer
from transformers import Trainer

trainer_b_exp2 = Trainer(
    model=model_b_exp2,
    args=training_args_b_exp2,

    train_dataset=qa_train_features,
    eval_dataset=qa_val_features,

    data_collator=data_collator
)

#train
train_output_b_exp2 = trainer_b_exp2.train()

Epoch,Training Loss,Validation Loss
1,5.600069,5.001671
2,4.310943,3.342358
3,3.011074,2.462213
4,2.287791,1.983243
5,1.762094,1.612820
6,1.492676,1.363118
7,1.251547,1.224844
8,1.139903,1.173858


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [72]:
#checking
print("Best checkpoint:", trainer_b_exp2.state.best_model_checkpoint)
print("Best validation loss:", trainer_b_exp2.state.best_metric)
print("Final training loss:", train_output_b_exp2.training_loss)

Best checkpoint: models/qa_model_exp2/checkpoint-56
Best validation loss: 1.173857569694519
Final training loss: 2.607011999402727


## 7. Fine-Tuned Model Evaluation

The best fine-tuned checkpoint is evaluated on the same held-out test
set used for the baseline.


In [73]:
fine_tuned_model_b = trainer_b_exp2.model
fine_tuned_model_b.to(device)
fine_tuned_model_b.eval()

print("Fine-tuned Model B is ready for evaluation.")

Fine-tuned Model B is ready for evaluation.


In [74]:
#to predict on test set
fine_tuned_predictions = []

for _, row in qa_test_df.iterrows():

    prediction = predict_answer(
        fine_tuned_model_b,
        row["question"],
        row["context"]
    )

    fine_tuned_predictions.append({
        "issue": row["issue"],
        "question": row["question"],
        "reference": row["answer_text"],
        "prediction": prediction,

        "exact_match": exact_match_score(
            prediction,
            row["answer_text"]
        ),

        "f1": token_f1_score(
            prediction,
            row["answer_text"]
        )
    })

In [75]:
#calculate Exact Match and F1
fine_tuned_results_df = pd.DataFrame(fine_tuned_predictions)
fine_tuned_em = fine_tuned_results_df["exact_match"].mean()
fine_tuned_f1 = fine_tuned_results_df["f1"].mean()

print("FINE-TUNED RESULTS")
print("-------------------------")
print(f"Exact Match : {fine_tuned_em:.4f}")
print(f"Token F1    : {fine_tuned_f1:.4f}")

FINE-TUNED RESULTS
-------------------------
Exact Match : 0.5000
Token F1    : 0.6331


In [76]:
#show all 8 predictions
fine_tuned_results_df[
    [
        "question",
        "reference",
        "prediction",
        "exact_match",
        "f1"
    ]
]

,question,reference,prediction,exact_match,f1
0,What are the symptoms of Wi-Fi not connecting?,"No internet, error messages","Restart router, reset adapter, check driver settings",0,0.000000
1,How can Wi-Fi not connecting be resolved?,"Restart router, reset adapter, check driver settings","Restart router, reset adapter, check driver settings",1,1.000000
2,What are the symptoms of Laptop overheating?,"Fan running loud, sudden shutdowns","Fan running loud, sudden shut",0,0.800000
3,How can Laptop overheating be resolved?,"Clean vents, apply thermal paste, use a cooling pad","Clean vents, apply thermal paste, use a cooling pad",1,1.000000
4,What are the symptoms of AirDrop not working?,Devices not visible or unable to send files,"Check Bluetooth and Wi-Fi, ensure devices are discoverable, restart devices",0,0.111111
5,How can AirDrop not working be resolved?,"Check Bluetooth and Wi-Fi, ensure devices are discoverable, restart devices","Check Bluetooth and Wi-Fi, ensure devices are discoverable, restart devices",1,1.000000
6,What are the symptoms of Outlook not sending emails?,Emails stuck in Outbox,"Check internet connection, clear Outbox, repair email account settings",0,0.153846
7,How can Outlook not sending emails be resolved?,"Check internet connection, clear Outbox, repair email account settings","Check internet connection, clear Outbox, repair email account settings",1,1.000000


In [77]:
#comparsion with the baseline
comparison_b = pd.DataFrame({
    "Metric": [
        "Exact Match",
        "Token F1"
    ],

    "Baseline": [
        baseline_em,
        baseline_f1
    ],

    "Fine-Tuned": [
        fine_tuned_em,
        fine_tuned_f1
    ]
})

comparison_b["Improvement"] = (
    comparison_b["Fine-Tuned"]
    - comparison_b["Baseline"]
)

comparison_b

,Metric,Baseline,Fine-Tuned,Improvement
0,Exact Match,0.000000,0.50000,0.500000
1,Token F1,0.416223,0.63312,0.216897


###Experiment 3 — 15 epochs

In [78]:
from transformers import AutoModelForQuestionAnswering, set_seed
set_seed(42)

model_b_exp3 = AutoModelForQuestionAnswering.from_pretrained(MODEL_B)
model_b_exp3.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

In [79]:
#training config
from transformers import TrainingArguments

training_args_b_exp3 = TrainingArguments(
    output_dir="models/qa_model_exp3",

    learning_rate=3e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=15,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_strategy="epoch",

    report_to="none",
    seed=42
)

In [81]:
#trainer and train
from transformers import Trainer

trainer_b_exp3 = Trainer(
    model=model_b_exp3,
    args=training_args_b_exp3,

    train_dataset=qa_train_features,
    eval_dataset=qa_val_features,

    data_collator=data_collator
)

train_output_b_exp3 = trainer_b_exp3.train()

Epoch,Training Loss,Validation Loss
1,5.593691,4.963996
2,4.221071,3.192885
3,2.842269,2.287589
4,2.055628,1.684268
5,1.433351,1.237380
6,1.083824,0.926075
7,0.752720,0.737388
8,0.596953,0.518944
9,0.417069,0.345288
10,0.341443,0.219022


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [82]:
#checking
print("Best checkpoint:",trainer_b_exp3.state.best_model_checkpoint)
print("Best validation loss:",trainer_b_exp3.state.best_metric)
print("Final training loss:",train_output_b_exp3.training_loss)

Best checkpoint: models/qa_model_exp3/checkpoint-105
Best validation loss: 0.07575536519289017
Final training loss: 1.3308418069566998


In [83]:
# use the best model from Experiment 3
fine_tuned_model_b = trainer_b_exp3.model

fine_tuned_model_b.to(device)
fine_tuned_model_b.eval()

print("Experiment 3 model is ready for evaluation.")

Experiment 3 model is ready for evaluation.


In [84]:
fine_tuned_predictions_exp3 = []

for _, row in qa_test_df.iterrows():

    prediction = predict_answer(
        fine_tuned_model_b,
        row["question"],
        row["context"]
    )

    fine_tuned_predictions_exp3.append({
        "issue": row["issue"],
        "question": row["question"],
        "reference": row["answer_text"],
        "prediction": prediction,

        "exact_match": exact_match_score(
            prediction,
            row["answer_text"]
        ),

        "f1": token_f1_score(
            prediction,
            row["answer_text"]
        )
    })

In [85]:
#calculate exact match and F1
exp3_results_df = pd.DataFrame(fine_tuned_predictions_exp3)
exp3_em = exp3_results_df["exact_match"].mean()
exp3_f1 = exp3_results_df["f1"].mean()

print("EXPERIMENT 3 TEST RESULTS")
print("----------------------------")
print(f"Exact Match : {exp3_em:.4f}")
print(f"Token F1    : {exp3_f1:.4f}")

EXPERIMENT 3 TEST RESULTS
----------------------------
Exact Match : 1.0000
Token F1    : 1.0000


In [86]:
#show predictions
exp3_results_df[
    [
        "question",
        "reference",
        "prediction",
        "exact_match",
        "f1"
    ]
]

,question,reference,prediction,exact_match,f1
0,What are the symptoms of Wi-Fi not connecting?,"No internet, error messages","No internet, error messages",1,1.0
1,How can Wi-Fi not connecting be resolved?,"Restart router, reset adapter, check driver settings","Restart router, reset adapter, check driver settings",1,1.0
2,What are the symptoms of Laptop overheating?,"Fan running loud, sudden shutdowns","Fan running loud, sudden shutdowns",1,1.0
3,How can Laptop overheating be resolved?,"Clean vents, apply thermal paste, use a cooling pad","Clean vents, apply thermal paste, use a cooling pad",1,1.0
4,What are the symptoms of AirDrop not working?,Devices not visible or unable to send files,Devices not visible or unable to send files,1,1.0
5,How can AirDrop not working be resolved?,"Check Bluetooth and Wi-Fi, ensure devices are discoverable, restart devices","Check Bluetooth and Wi-Fi, ensure devices are discoverable, restart devices",1,1.0
6,What are the symptoms of Outlook not sending emails?,Emails stuck in Outbox,Emails stuck in Outbox,1,1.0
7,How can Outlook not sending emails be resolved?,"Check internet connection, clear Outbox, repair email account settings","Check internet connection, clear Outbox, repair email account settings",1,1.0


In [87]:
#show full comparision
model_b_comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Exp 2 - 8 epochs",
        "Exp 3 - 15 epochs"
    ],

    "Exact Match": [
        baseline_em,
        fine_tuned_em,
        exp3_em
    ],

    "Token F1": [
        baseline_f1,
        fine_tuned_f1,
        exp3_f1
    ]
})

model_b_comparison

,Model,Exact Match,Token F1
0,Baseline,0.0,0.416223
1,Exp 2 - 8 epochs,0.5,0.633120
2,Exp 3 - 15 epochs,1.0,1.000000


### **Experiment Findings**

| Experiment | Epochs | Best Validation Loss | Test Exact Match | Test Token F1 | Finding |
|---|---:|---:|---:|---:|---|
| Exp 1 | 2 | 4.7254 | Not evaluated as final | Not evaluated as final | The model was clearly underfitting. Both training and validation loss were still high and decreasing, so 2 epochs were not enough. |
| Exp 2 | 8 | 1.1739 | 0.5000 | 0.6331 | Training improved substantially. The model learned resolution questions well, but still confused some symptom questions with solution spans. |
| Exp 3 | 15 | 0.0758 | 1.0000 | 1.0000 | Longer training resolved the remaining extraction errors and the model correctly answered all QA pairs in the held-out test set. |

### Exp 1

- Training loss decreased from **5.6393** to **4.9378**.
- Validation loss decreased from **5.2225** to **4.7254**.
- The losses were still high after 2 epochs.
- No sign of overfitting was observed.
- Conclusion: **2 epochs were insufficient**, so training was extended.

### Exp 2

- Training was increased to **8 epochs**.
- Validation loss improved continuously from **5.0017** to **1.1739**.
- Best checkpoint: **checkpoint-56**.
- Fine-tuned test results:
  - **Exact Match: 0.5000**
  - **Token F1: 0.6331**
- Compared with the baseline:
  - EM improved from **0.0000 → 0.5000**
  - F1 improved from **0.4162 → 0.6331**
- Most resolution questions were answered correctly.
- Remaining errors mainly occurred when the model confused **symptom questions** with **solution spans**.

### Exp 3

- Training was extended to **15 epochs** to address the remaining underfitting.
- Training loss decreased from **5.5937** to **0.0715**.
- Validation loss decreased from **4.9640** to **0.0758**.
- Best checkpoint: **checkpoint-105**.
- Best validation loss: **0.0758**.
- Test results:
  - **Exact Match: 1.0000**
  - **Token F1: 1.0000**
- All **8 held-out QA pairs** were answered exactly correctly.
- The model successfully learned to distinguish between **symptom-related questions** and **resolution-related questions**.
- Exp 3 passed the required quality gate:
  - EM ≥ **0.65**
  - F1 ≥ **0.80**

### Finding

Increasing the training duration from 2 to 8 and finally 15 epochs produced a clear improvement in Model B.

The model progressed from underfitting in Exp 1, to partial answer-span extraction in Exp 2, and finally to exact answer extraction in Exp 3.

Exp 3 achieved **1.0000 Exact Match** and **1.0000 Token F1** on the current held-out test set.

The 100% result should be interpreted specifically for this small project test set of **8 QA pairs from 4 unseen IT issues**, rather than as a general real-world accuracy estimate.

## 8. Quality Gate & Final Evaluation


In [88]:
EM_THRESHOLD = 0.65
F1_THRESHOLD = 0.80

em_pass = exp3_em >= EM_THRESHOLD
f1_pass = exp3_f1 >= F1_THRESHOLD

model_b_gate_passed = em_pass and f1_pass

print("MODEL B QUALITY GATE")
print("-------------------------")
print(f"Exact Match : {exp3_em:.4f}  | Required: {EM_THRESHOLD}")
print(f"Token F1    : {exp3_f1:.4f}  | Required: {F1_THRESHOLD}")

print(
    "\nRESULT:",
    "PASSED" if model_b_gate_passed else "FAILED"
)

MODEL B QUALITY GATE
-------------------------
Exact Match : 1.0000  | Required: 0.65
Token F1    : 1.0000  | Required: 0.8

RESULT: PASSED


##9. Model Saving & Final Results

In [89]:
#save the best model
MODEL_B_SAVE_PATH = "models/qa_model"
trainer_b_exp3.model.save_pretrained(MODEL_B_SAVE_PATH)
tokenizer_b.save_pretrained(MODEL_B_SAVE_PATH)

print("Model B saved to:", MODEL_B_SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model B saved to: models/qa_model


In [90]:
#reload check
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering
)

saved_tokenizer_b = AutoTokenizer.from_pretrained(
    MODEL_B_SAVE_PATH
)

saved_model_b = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_B_SAVE_PATH
)

print("Model B reloaded successfully.")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model B reloaded successfully.


In [93]:
#save the final QA dataset
import os

# create the data folder if it does not already exist
os.makedirs("data", exist_ok=True)

# save the final QA dataset
qa_df[
    [
        "issue",
        "question",
        "context",
        "answer_text",
        "answer_start"
    ]
].to_json(
    "data/qa_train.json",
    orient="records",
    indent=2
)

print("Saved: data/qa_train.json")

Saved: data/qa_train.json


In [94]:
#save the evaluation report
import json
import os

os.makedirs("reports", exist_ok=True)

model_b_results = {
    "model": MODEL_B,

    "dataset": {
        "unique_it_issues": 33,
        "total_qa_pairs": 66,
        "train_pairs": 52,
        "validation_pairs": 6,
        "test_pairs": 8
    },

    "baseline": {
        "exact_match": float(baseline_em),
        "token_f1": float(baseline_f1)
    },

    "experiment_2": {
        "epochs": 8,
        "exact_match": float(fine_tuned_em),
        "token_f1": float(fine_tuned_f1)
    },

    "experiment_3": {
        "epochs": 15,
        "best_validation_loss": float(
            trainer_b_exp3.state.best_metric
        ),
        "exact_match": float(exp3_em),
        "token_f1": float(exp3_f1)
    },

    "quality_gate": {
        "required_em": 0.65,
        "required_f1": 0.80,
        "passed": bool(model_b_gate_passed)
    }
}

with open(
    "reports/model_b_results.json",
    "w"
) as f:
    json.dump(
        model_b_results,
        f,
        indent=4
    )

print("Saved: reports/model_b_results.json")

Saved: reports/model_b_results.json


In [95]:
#push Model B to HuggingFace
HF_MODEL_B_REPO = ("JoudAlrubaish/technical-support-extractive-qa")

saved_model_b.push_to_hub(
    HF_MODEL_B_REPO,
    token=hf_token
)

saved_tokenizer_b.push_to_hub(
    HF_MODEL_B_REPO,
    token=hf_token
)

print(f"Uploaded Model B to: {HF_MODEL_B_REPO}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...eube_za/model.safetensors:   3%|2         | 6.68MB /  265MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Uploaded Model B to: JoudAlrubaish/technical-support-extractive-qa


In [108]:
!ls data
!ls reports

golden_set.jsonl  intents.csv  qa_train.json  sft_train.jsonl
model_a_results.json  model_b_results.json


In [110]:
import os

print(
    "Repo exists:",
    os.path.isdir("/content/technical-support-agent/.git")
)

Repo exists: True


In [111]:
%cd /content/technical-support-agent
!git status

/content/technical-support-agent
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [112]:
import os
import shutil

os.makedirs("data", exist_ok=True)
os.makedirs("reports", exist_ok=True)

shutil.copy(
    "/content/data/qa_train.json",
    "data/qa_train.json"
)

shutil.copy(
    "/content/reports/model_b_results.json",
    "reports/model_b_results.json"
)

print("Model B files copied.")

Model B files copied.


In [113]:
!git config user.name "Joud Alrubaish"
!git config user.email "joudalrubaish2@gmail.com"

!git add data/qa_train.json reports/model_b_results.json
!git commit -m "Add Model B QA dataset and evaluation results"

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [115]:
%cd /content/technical-support-agent

/content/technical-support-agent


In [116]:
!git remote set-url origin https://github.com/JoudAlrubaish/technical-support-agent.git

In [117]:
from google.colab import userdata
import subprocess
import os

github_token = userdata.get("GITHUB_TOKEN")

askpass_path = "/tmp/git_askpass.sh"

with open(askpass_path, "w") as f:
    f.write("""#!/bin/sh
case "$1" in
    *Username*) echo "JoudAlrubaish" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")

os.chmod(askpass_path, 0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = github_token
env["GIT_ASKPASS"] = askpass_path
env["GIT_TERMINAL_PROMPT"] = "0"

result = subprocess.run(
    ["git", "push", "origin", "main"],
    env=env,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ Model B files pushed to GitHub.")
else:
    print("Push failed:")
    print(result.stderr)

✅ Model B files pushed to GitHub.
